In [0]:
%python

import sys
sys.path.append("/Workspace/Users/alliesdl@gmail.com/credit_risk_pipeline")  # your Git folder

from pyspark.sql import functions as F, Window
from src.quality import assert_unique

cust = spark.table("lender_portfolio.silver.customers")
apps = spark.table("lender_portfolio.silver.applications")
loans = spark.table("lender_portfolio.silver.loans")
reps = spark.table("lender_portfolio.silver.repayments")
parts = spark.table("lender_portfolio.silver.partners")

# 1. How many applications did this customer make before this one? (known at application time)
w = (Window.partitionBy("customer_id")
     .orderBy("application_date", "application_id")
     .rowsBetween(Window.unboundedPreceding, -1))
apps = apps.withColumn("n_prior_applications", F.count("*").over(w))

# 2. The label: 90+ days past due within the first 6 instalments (or fewer for a short loan)
window_len = F.least(F.col("term_months"), F.lit(6))
obs = (reps.join(loans.select("loan_id", "term_months"), "loan_id")
       .filter(F.col("instalment_no") <= window_len)
       .groupBy("loan_id")
       .agg(F.max("days_past_due").alias("max_dpd"),
            F.count("*").alias("n_observed")))

# 3. One row per loan, application-time information only
g = (loans.alias("l")
     .join(apps.select("application_id", "application_date", "partner_id", "n_prior_applications"),
           "application_id")
     .join(cust.drop("signup_date", "signup_date_raw"), "customer_id", "left")
     .join(parts.select("partner_id", "channel_type"), "partner_id", "left")
     .join(obs, "loan_id", "left"))

instalment = (F.col("principal")
              * (1 + F.col("interest_rate_pct") / 100 * F.col("term_months") / 12)
              / F.col("term_months"))

g = (g.withColumn("age_at_application", F.year("application_date") - F.col("birth_year"))
      .withColumn("instalment", instalment)
      .withColumn("dti", F.expr("try_divide(instalment, monthly_income)"))
      .withColumn("loan_to_income", F.expr("try_divide(principal, monthly_income)"))
      .withColumn("application_month", F.trunc("application_date", "month")))

# 4. Label only for loans we have watched for the whole window. Keeping a loan that already
#    defaulted but dropping one that has not yet would inflate the recent default rate.
defaulted = F.col("max_dpd") >= 90
matured = F.col("n_observed") >= window_len

g = (g.withColumn("matured", F.coalesce(matured, F.lit(False)))
      .withColumn("default_flag", F.when(F.col("matured"), defaulted.cast("int")))
      .drop("max_dpd", "n_observed", "birth_year", "instalment", "disbursement_date_raw"))

assert_unique(g, "loan_id")

(g.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.gold.loan_table"))

print(g.count(),
      g.groupBy("matured", "default_flag").count()
       .orderBy("matured", "default_flag").collect())

In [0]:
%python

import sys
sys.path.append("/Workspace/Users/alliesdl@gmail.com/credit_risk_pipeline") # your Git folder

from pyspark.sql import functions as F, Window
from src.quality import assert_unique

apps = spark.table("lender_portfolio.silver.applications")
camp = spark.table("lender_portfolio.silver.campaigns")

# 1. Attribute each application to the campaign running for its partner on its date.
#    If two campaigns overlap, the one that started most recently gets it.
j = (apps.alias("a")
     .join(camp.alias("c"),
           (F.col("a.partner_id") == F.col("c.partner_id")) &
           (F.col("a.application_date") >= F.col("c.start_date")) &
           (F.col("a.application_date") <= F.col("c.end_date")),
           "left")
     .select("a.application_id", "a.partner_id", "a.application_date",
             "a.decision", "c.campaign_id", "c.start_date"))

w = Window.partitionBy("application_id").orderBy(F.col("start_date").desc_nulls_last())

attr = (j.withColumn("rn", F.row_number().over(w))
         .filter("rn = 1")
         .drop("rn", "start_date"))

assert attr.count() == apps.count(), "attribution must not add or lose applications"

(attr.write.format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("lender_portfolio.gold.application_campaign"))

# 2. Per campaign: what did it cost per application and per approved loan?
camp_eff = (attr.filter("campaign_id IS NOT NULL")
            .groupBy("campaign_id")
            .agg(F.count("*").alias("applications"),
                 F.sum((F.col("decision") == "Approved").cast("int")).alias("loans"))
            .join(camp.select("campaign_id", "partner_id", "campaign_name",
                              "start_date", "end_date", "spend_zar"),
                  "campaign_id", "right")
            .fillna(0, ["applications", "loans"])
            .withColumn("cost_per_application", F.expr("try_divide(spend_zar, applications)"))
            .withColumn("cost_per_loan", F.expr("try_divide(spend_zar, loans)")))

assert_unique(camp_eff, "campaign_id")

(camp_eff.write.format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable("lender_portfolio.gold.campaign_efficiency"))

organic = attr.filter("campaign_id IS NULL").count()
print("applications with no campaign (organic):", organic, "of", apps.count())
print("campaign_efficiency rows:", camp_eff.count())